[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfc/janus-core/blob/main/docs/source/tutorials/cli/eos.ipynb)

# Equation of State

## Set up environment (optional)

These steps are required to run this tutorial with Google Colab. To do so, uncomment and run the cell below.

This will replace pre-installed versions of `numpy` and `torch` in Colab with versions that are known to be compatible with `janus-core`.

It may be possible to skip the steps that uninstall and reinstall `torch`, which will save a considerable amount of time.

These instructions but may work for other systems too, but it is typically preferable to prepare a virtual environment separately before running this notebook if possible.

In [ ]:
# import locale
# locale.getpreferredencoding = lambda: "UTF-8"

# ! pip uninstall numpy -y # Uninstall pre-installed numpy

# ! pip uninstall torch torchaudio torchvision transformers -y # Uninstall pre-installed torch
# ! uv pip install torch==2.5.1 # Install pinned version of torch

# ! uv pip install janus-core[mace,visualise] data-tutorials --system # Install janus-core with MACE and WeasWidget, and data-tutorials

# get_ipython().kernel.do_shutdown(restart=True) # Restart kernel to update libraries. This may warn that your session has crashed.

To ensure you have the latest version of `janus-core` installed, compare the output of the following cell to the latest version available at https://pypi.org/project/janus-core/

In [ ]:
from janus_core import __version__

print(__version__)

## Prepare data

Use `data_tutorials` to get the data required for this tutorial:

In [ ]:
from data_tutorials.data import get_data

get_data(
    url="https://raw.githubusercontent.com/stfc/janus-core/main/docs/source/tutorials/data/",
    filename=["beta_quartz.cif"],
    folder="data",
)

In [ ]:
from pathlib import Path

from ase import Atoms
from ase.io import write
from weas_widget import WeasWidget
Path("../data").mkdir(exist_ok=True)

α_quartz = Atoms(
    symbols=(*["Si"] * 3, *["O"] * 6),
    scaled_positions=[
        [0.469700, 0.000000, 0.000000],
        [0.000000, 0.469700, 0.666667],
        [0.530300, 0.530300, 0.333333],
        [0.413500, 0.266900, 0.119100],
        [0.266900, 0.413500, 0.547567],
        [0.733100, 0.146600, 0.785767],
        [0.586500, 0.853400, 0.214233],
        [0.853400, 0.586500, 0.452433],
        [0.146600, 0.733100, 0.880900],
    ],
    cell=[
        [4.916000, 0.000000, 0.000000],
        [-2.45800, 4.257381, 0.000000],
        [0.000000, 0.000000, 5.405400],
    ],
    pbc=True,
)

write("../data/α_quartz.xyz", α_quartz)

v=WeasWidget()
v.from_ase(α_quartz)
v.avr.model_style = 1
v.avr.show_hydrogen_bonds = True
v

## Command-line help and options

Once `janus-core` is installed, the `janus` CLI command should be available:

In [ ]:
! janus eos --help

## Equation of state for α-quartz

In [ ]:
%%writefile eos.yml

struct: ../data/α_quartz.xyz
device: cpu
arch: mace_mp
model: medium-omat-0
min_volume: 0.75
max_volume: 1.25
n_volumes: 20
plot_to_file: True
write_structures: True
minimize: False
tracker: False

In [ ]:
!janus eos --config eos.yml

In [ ]:
!ls janus_results/

Since we saved the equation of state plot and generated structures, we can visualise these:

In [ ]:
from IPython.display import SVG, display
display(SVG("janus_results/α_quartz-eos-plot.svg"))

In [ ]:
from ase.io import read

α_quartz_eos = read("janus_results/α_quartz-generated.extxyz", index=":")

w=WeasWidget()
w.from_ase(α_quartz_eos)
w.avr.model_style = 1
w.avr.show_hydrogen_bonds = True
w

## Equation of state for β-quartz

Perform the same calculation for β-quartz:

In [ ]:
β_quartz = read("data/beta_quartz.cif")

v=WeasWidget()
v.from_ase(β_quartz)
v.avr.model_style = 1
v.avr.show_hydrogen_bonds = True
v

Here we also add to options to run geometry optimisations. ``--minimize`` performs a relaxation on the initial β-quartz structure, including allowing the cell to be fully optimised, while ``--minimize-all`` relaxes the scaled structures that are generated, but does so at constant volume automatically.

In [ ]:
!janus eos --config eos.yml --struct ../data/beta_quartz.cif --minimize --minimize-all

We can now visulise these as before:

In [ ]:
display(SVG("janus_results/beta_quartz-eos-plot.svg"))

In [ ]:
beta_quartz_eos = read("janus_results/beta_quartz-generated.extxyz", index=":")

w=WeasWidget()
w.from_ase(beta_quartz_eos)
w.avr.model_style = 1
w.avr.show_hydrogen_bonds = True
w